# Figure Summarizer

Extract figures from a scientific PDF, pair them with their captions,
and generate summaries using a vision-language model (BLIP-2).

The context window for each prediction is **only the figure image + its caption** — no full paper text.

In [1]:
import sys
from pathlib import Path

# Add project root to path
PROJECT_ROOT = Path.cwd().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import config
print(f"Device: {config.DEVICE}")
print(f"Model:  {config.VLM_MODEL_ID}")

Device: mps
Model:  Salesforce/blip2-opt-2.7b


## 1. Pick a paper

In [2]:
# Point this at any PDF
PAPER_PATH = config.DEFAULT_PAPERS_DIR / "1-s2.0-S0042698925001154-main.pdf"

# Or browse and pick:
# import glob
# pdfs = sorted(config.DEFAULT_PAPERS_DIR.glob("*.pdf"))
# for i, p in enumerate(pdfs):
#     print(f"  [{i}] {p.name}")
# PAPER_PATH = pdfs[0]

print(f"Paper: {PAPER_PATH.name}")

Paper: 1-s2.0-S0042698925001154-main.pdf


## 2. Extract figures and captions

In [3]:
from extract import extract_figures_from_pdf

extraction = extract_figures_from_pdf(PAPER_PATH)

print(f"\nExtracted {len(extraction.figures)} figures from {extraction.num_pages} pages")
for fig in extraction.figures:
    caption_preview = fig.caption[:100] + "..." if len(fig.caption) > 100 else fig.caption
    print(f"  {fig.figure_id}  (page {fig.page_number + 1}, {fig.width}x{fig.height}px)")
    print(f"    Caption: {caption_preview}")

Extracted 6 figures from 1-s2.0-S0042698925001154-main.pdf

Extracted 6 figures from 9 pages
  fig_1  (page 1, 248x271px)
    Caption: 
  fig_2  (page 1, 236x298px)
    Caption: 
  fig_3  (page 2, 1896x1052px)
    Caption: Fig. 1. Retinal receptive fields increase in complexity throughout the retinal circuit. A) A schemat...
  fig_4  (page 3, 2129x752px)
    Caption: Fig. 2. Bipolar cell receptive fields are diverse in their processing of spatial and temporal visual...
  fig_5  (page 5, 864x2602px)
    Caption: 
  fig_6  (page 7, 1421x434px)
    Caption: Fig. 4. Neuromodulators alter retinal receptive fields. Neuromodulator-releasing neurons are schemat...


## 3. Preview extracted figures

In [4]:
from PIL import Image
import matplotlib.pyplot as plt

n_figs = len(extraction.figures)
if n_figs > 0:
    cols = min(n_figs, 3)
    rows = (n_figs + cols - 1) // cols
    fig, axes = plt.subplots(rows, cols, figsize=(6 * cols, 5 * rows))
    if n_figs == 1:
        axes = [axes]
    else:
        axes = axes.flatten()

    for i, efig in enumerate(extraction.figures):
        img = Image.open(efig.image_path)
        axes[i].imshow(img)
        axes[i].set_title(f"{efig.figure_id} (p{efig.page_number + 1})", fontsize=10)
        axes[i].axis("off")

    # Hide unused subplots
    for j in range(i + 1, len(axes)):
        axes[j].axis("off")

    plt.tight_layout()
    plt.show()
else:
    print("No figures to preview.")

ModuleNotFoundError: No module named 'matplotlib'

## 4. Load the vision-language model

In [ ]:
from model import FigureSummarizer

summarizer = FigureSummarizer()

## 5. Summarize each figure

Each figure is passed to the model **individually** — the context window
contains only that figure's image and its caption. No paper text.

In [ ]:
summaries = []

for efig in extraction.figures:
    print(f"\n{'='*60}")
    print(f"{efig.figure_id}  (page {efig.page_number + 1})")
    print(f"{'='*60}")

    if efig.caption:
        print(f"Caption: {efig.caption[:200]}...\n")

    summary = summarizer.summarize(efig.image_path, caption=efig.caption)
    summaries.append({
        "figure_id": efig.figure_id,
        "page": efig.page_number + 1,
        "caption": efig.caption,
        "summary": summary,
        "image_path": efig.image_path,
    })

    print(f"Summary:\n{summary}")

## 6. Display results side-by-side

In [ ]:
from IPython.display import display, HTML, Markdown

for s in summaries:
    display(Markdown(f"### {s['figure_id']} (page {s['page']})"))

    img = Image.open(s["image_path"])
    img.thumbnail((600, 500))

    fig_disp, ax = plt.subplots(1, 1, figsize=(8, 6))
    ax.imshow(img)
    ax.axis("off")
    plt.tight_layout()
    plt.show()

    if s["caption"]:
        display(Markdown(f"**Caption:** {s['caption']}"))
    display(Markdown(f"**Summary:** {s['summary']}"))
    display(Markdown("---"))

## 7. Save results

In [ ]:
import json

output_dir = config.OUTPUT_DIR / PAPER_PATH.stem
output_dir.mkdir(parents=True, exist_ok=True)

# Save JSON
json_path = output_dir / "summaries.json"
json_path.write_text(json.dumps(summaries, indent=2, ensure_ascii=False))
print(f"Saved JSON:     {json_path}")

# Save Markdown
md_lines = [f"# Figure Summaries: {PAPER_PATH.name}\n"]
for s in summaries:
    md_lines.append(f"## {s['figure_id']} (page {s['page']})\n")
    if s["caption"]:
        md_lines.append(f"**Caption:** {s['caption']}\n")
    md_lines.append(f"**Summary:** {s['summary']}\n")
    md_lines.append("---\n")

md_path = output_dir / "summaries.md"
md_path.write_text("\n".join(md_lines))
print(f"Saved Markdown: {md_path}")

print(f"\nAll outputs in: {output_dir}")

## 8. (Optional) Summarize a single figure interactively

Pass any image + custom prompt to the model.

In [ ]:
# Pick a specific figure to re-summarize with a custom prompt
fig_idx = 0  # change this to pick a different figure

if extraction.figures:
    target = extraction.figures[fig_idx]
    img = Image.open(target.image_path)
    plt.imshow(img)
    plt.axis("off")
    plt.title(target.figure_id)
    plt.show()

    # Custom question about the figure
    custom_summary = summarizer.summarize(
        target.image_path,
        caption="What are the main trends or patterns visible in this figure?",
    )
    print(f"\nCustom summary:\n{custom_summary}")